# inplace-param-update — faded example 1: Fill the in-place SGD update line

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-param-update`. The last cell reports your progress on the `PyTorch: In-place param update` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: In-place param update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-param-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-param-update"
DD_SUBTOPIC = "PyTorch: In-place param update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A hand-rolled SGD step must mutate each parameter's storage in place so the model and optimizer keep seeing the same tensor object. `p.data.sub_(g, alpha=lr)` computes `p.data -= lr * g` without allocating a new tensor, preserving `data_ptr()`.

## Faded exercise 1

Complete `sgd_step(params, grads, lr)`. It should update every parameter in place so that each parameter's `data_ptr()` is identical before and after. Fill in the single in-place update line.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(3)

def sgd_step(params, grads, lr):
    for p, g in zip(params, grads):
        p.data.sub_(g, alpha=lr)
    return params


def _test():
    lin = nn.Linear(3, 2)
    params = [lin.weight, lin.bias]
    grads = [t.full_like(p, 2.0) for p in params]
    before = [p.detach().clone() for p in params]
    ptrs = [p.data_ptr() for p in params]
    out = sgd_step(params, grads, lr=0.5)
    assert out is params
    for p, b, ptr in zip(params, before, ptrs):
        # independent ground truth: p_new = p_old - lr*grad = b - 0.5*2 = b - 1
        assert t.allclose(p.detach(), b - 1.0)
        assert p.data_ptr() == ptr, 'storage must be preserved (in-place)'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

t.manual_seed(3)

def sgd_step(params, grads, lr):
    for p, g in zip(params, grads):
        p.data.sub_(g, alpha=lr)
    return params
```
</details>